# SQL Injection Detector - Complete Colab Notebook

This notebook contains everything you need to train and use the SQL Injection Detector.

## 📋 Instructions

1. **Upload your dataset** to `/content/data/` (see instructions below)
2. **Enable GPU**: Runtime → Change runtime type → GPU (required for Stage 2)
3. **Run all cells** sequentially

## 📊 Dataset Format

Create two files in `/content/data/`:
- `safe_queries.txt` - Normal SQL queries (one per line)
- `injection_queries.txt` - SQL injection attacks (one per line)

Or use CSV/JSON format (see COLAB_INSTRUCTIONS.md)

## Step 1: Setup Environment

In [ ]:
# Install all dependencies
!pip install -q numpy scikit-learn transformers torch mmh3 pyyaml onnxruntime bitarray

In [ ]:
# Setup paths and import system
import os
import sys
from pathlib import Path

# Set working directory to colab_ready folder
# Adjust this path if you uploaded to a different location
if os.path.exists('/content/colab_ready'):
    os.chdir('/content/colab_ready')
    sys.path.insert(0, '/content/colab_ready')
    print(f"✓ Working directory: {os.getcwd()}")
elif os.path.exists('/content/drive/MyDrive/colab_ready'):
    os.chdir('/content/drive/MyDrive/colab_ready')
    sys.path.insert(0, '/content/drive/MyDrive/colab_ready')
    print(f"✓ Working directory: {os.getcwd()}")
else:
    print("⚠️ colab_ready folder not found!")
    print("Please upload the colab_ready folder to /content/ or Google Drive")
    print("\nTo upload:")
    print("1. Click folder icon on left sidebar")
    print("2. Click upload button")
    print("3. Upload the entire colab_ready folder")

In [ ]:
# Verify installation and check GPU
import torch
import transformers
import sklearn
import mmh3

print("✓ All dependencies installed successfully!")
print(f"\nPyTorch version: {torch.__version__}")
print(f"Transformers version: {transformers.__version__}")
print(f"\nCUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"✓ GPU: {torch.cuda.get_device_name(0)}")
    print(f"CUDA version: {torch.version.cuda}")
else:
    print("⚠️ GPU not available. Stage 2 training will be slow.")
    print("Go to: Runtime → Change runtime type → GPU")

## Step 2: Prepare Dataset

### Option A: Upload Your Dataset

Upload your dataset files to `/content/data/`:
- `safe_queries.txt`
- `injection_queries.txt`

### Option B: Use Sample Data (For Testing)

If you don't have data yet, we'll create sample data below.


In [ ]:
# Create data directory
os.makedirs('/content/data', exist_ok=True)

# Check if you already have data
safe_file = Path('/content/data/safe_queries.txt')
injection_file = Path('/content/data/injection_queries.txt')

if safe_file.exists() and injection_file.exists():
    print("✓ Found your dataset files!")
    with open(safe_file, 'r') as f:
        safe_count = len([l for l in f if l.strip()])
    with open(injection_file, 'r') as f:
        injection_count = len([l for l in f if l.strip()])
    print(f"  Safe queries: {safe_count}")
    print(f"  Injection queries: {injection_count}")
    USE_SAMPLE_DATA = False
else:
    print("⚠️ Dataset not found. Creating sample data for testing...")
    print("\nTo use your own data:")
    print("1. Upload safe_queries.txt and injection_queries.txt to /content/data/")
    print("2. Re-run this cell")
    USE_SAMPLE_DATA = True

In [ ]:
# Create sample data if needed
if USE_SAMPLE_DATA:
    print("Creating sample training data...")
    
    # Sample safe queries
    safe_queries = [
        "SELECT * FROM users WHERE id = 1",
        "SELECT * FROM users WHERE name = 'John'",
        "SELECT * FROM products WHERE category = 'electronics'",
        "SELECT * FROM orders WHERE user_id = 100",
        "INSERT INTO users (name, email) VALUES ('Jane', 'jane@example.com')",
        "UPDATE users SET name = 'Bob' WHERE id = 5",
        "DELETE FROM orders WHERE order_id = 50",
        "SELECT COUNT(*) FROM users",
        "SELECT * FROM users ORDER BY name ASC",
        "SELECT u.name, o.total FROM users u JOIN orders o ON u.id = o.user_id",
        "SELECT * FROM products WHERE price BETWEEN 10 AND 100",
        "SELECT * FROM users WHERE email LIKE '%@gmail.com'",
        "SELECT DISTINCT category FROM products",
        "SELECT * FROM users LIMIT 10",
        "SELECT AVG(price) FROM products WHERE category = 'books'",
    ]
    
    # Sample injection queries
    injection_queries = [
        "SELECT * FROM users WHERE id = 1 OR 1=1",
        "SELECT * FROM users WHERE id = 1' OR '1'='1",
        "SELECT * FROM users UNION SELECT * FROM admin",
        "SELECT * FROM users WHERE name = 'admin'--",
        "SELECT * FROM users WHERE id = 1; DROP TABLE users;--",
        "SELECT * FROM users WHERE id = 1' UNION SELECT NULL, username, password FROM admin--",
        "SELECT * FROM users WHERE id = 1 OR 'a'='a",
        "SELECT * FROM users WHERE id = 1' AND '1'='1",
        "SELECT * FROM users WHERE id = 1' OR '1'='1'--",
        "SELECT * FROM users WHERE id = 1' OR '1'='1'/*",
        "SELECT * FROM users WHERE id = 1' UNION SELECT * FROM passwords--",
        "SELECT * FROM users WHERE id = 1' OR 1=1--",
        "SELECT * FROM users WHERE id = 1' OR 'x'='x",
        "SELECT * FROM users WHERE id = 1' OR 'a'='a'--",
        "SELECT * FROM users WHERE id = 1' OR '1'='1' LIMIT 1--",
    ]
    
    # Write to files
    with open('/content/data/safe_queries.txt', 'w') as f:
        f.write('\n'.join(safe_queries))
    
    with open('/content/data/injection_queries.txt', 'w') as f:
        f.write('\n'.join(injection_queries))
    
    print(f"✓ Created sample data:")
    print(f"  Safe queries: {len(safe_queries)}")
    print(f"  Injection queries: {len(injection_queries)}")
    print("\n⚠️ This is minimal sample data. For production, use 1000+ queries per class.")

## Step 3: Update Configuration

Update config.yaml to point to correct paths in Colab


In [ ]:
# Update config.yaml for Colab paths
import yaml

config_path = 'config.yaml'
if os.path.exists(config_path):
    with open(config_path, 'r') as f:
        config = yaml.safe_load(f)
    
    # Update model paths to /content/models
    os.makedirs('/content/models', exist_ok=True)
    
    config['stage0']['bloom_filter']['model_path'] = '/content/models/stage0_bloom.bin'
    config['stage1']['svm']['model_path'] = '/content/models/stage1_svm.pkl'
    config['stage2']['model']['model_path'] = '/content/models/stage2_distilbert'
    
    with open(config_path, 'w') as f:
        yaml.dump(config, f, default_flow_style=False)
    
    print("✓ Configuration updated for Colab")
else:
    print("⚠️ config.yaml not found")

In [ ]:
## Step 4: Train Stage 0 (Bloom Filter Whitelist)
# Train Stage 0
print("=" * 60)
print("Training Stage 0: Bloom Filter Whitelist")
print("=" * 60)

!python training/train_stage0.py --data /content/data --config config.yaml


In [ ]:
## Step 5: Train Stage 1 (Linear SVM)
# Train Stage 1
print("=" * 60)
print("Training Stage 1: Linear SVM Classifier")
print("=" * 60)

!python training/train_stage1.py --data /content/data --config config.yaml


In [ ]:
## Step 6: Train Stage 2 (DistilBERT) - Requires GPU
# Check GPU before training Stage 2
if not torch.cuda.is_available():
    print("⚠️ WARNING: GPU not available!")
    print("Stage 2 training will be VERY slow on CPU.")
    print("\nPlease enable GPU:")
    print("Runtime → Change runtime type → GPU → Save")
    print("\nThen re-run this cell.")
else:
    print(f"✓ GPU available: {torch.cuda.get_device_name(0)}")
    print("\nTraining Stage 2 (this may take 20-30 minutes)...")
    print("=" * 60)
    print("Training Stage 2: Quantized DistilBERT")
    print("=" * 60)
    
    !python training/train_stage2.py --data /content/data --config config.yaml --epochs 3


In [ ]:
## Step 7: Test the Detector
# Test the trained detector
from src.pipeline.detector import SQLInjectionDetector

print("Initializing SQL Injection Detector...")
detector = SQLInjectionDetector(config_path='config.yaml', metrics_enabled=True)
print("✓ Detector loaded!")

# Test queries
test_queries = [
    # Safe queries
    "SELECT * FROM users WHERE id = 1",
    "INSERT INTO users (name, email) VALUES ('John', 'john@example.com')",
    "UPDATE users SET name = 'Jane' WHERE id = 5",
    
    # Injection queries
    "SELECT * FROM users WHERE id = 1 OR 1=1",
    "SELECT * FROM users UNION SELECT * FROM admin",
    "SELECT * FROM users WHERE name = 'admin'--",
    "SELECT * FROM users WHERE id = 1; DROP TABLE users;--",
]

print("\n" + "=" * 60)
print("Testing SQL Injection Detection")
print("=" * 60)

for query in test_queries:
    result = detector.detect(query)
    
    status = "🚫" if result['decision'] == 'BLOCK' else "✅"
    print(f"\n{status} {result['decision']:5s} | Stage: {result['stage']:8s} | "
          f"Confidence: {result['confidence']:.4f}")
    print(f"   Query: {query}")
    if result['latencies']:
        latencies_str = ", ".join([f"{k}: {v:.2f}ms" for k, v in result['latencies'].items()])
        print(f"   Latencies: {latencies_str}")


In [ ]:
# Show performance metrics
metrics = detector.get_metrics_summary()

print("\n" + "=" * 60)
print("Performance Metrics")
print("=" * 60)

if metrics:
    pipeline_metrics = metrics.get('pipeline', {})
    print(f"Total queries processed: {pipeline_metrics.get('total_queries', 0)}")
    print(f"Allowed: {pipeline_metrics.get('total_allowed', 0)}")
    print(f"Blocked: {pipeline_metrics.get('total_blocked', 0)}")
    
    if 'stages' in metrics:
        print("\nStage Performance:")
        for stage_name, stage_metrics in metrics['stages'].items():
            print(f"\n  {stage_name}:")
            print(f"    Queries: {stage_metrics.get('total_queries', 0)}")
            print(f"    Avg Latency: {stage_metrics.get('avg_latency_ms', 0):.2f}ms")
            print(f"    Min Latency: {stage_metrics.get('min_latency_ms', 0):.2f}ms")
            print(f"    Max Latency: {stage_metrics.get('max_latency_ms', 0):.2f}ms")


In [ ]:
## Step 8: Save Models to Google Drive (Optional but Recommended)
# Mount Google Drive to save models
from google.colab import drive
drive.mount('/content/drive')

# Copy models to Drive for persistence
import shutil

drive_models_path = '/content/drive/MyDrive/sql_injection_models'
os.makedirs(drive_models_path, exist_ok=True)

if os.path.exists('/content/models'):
    # Copy entire models directory
    if os.path.exists(drive_models_path):
        shutil.rmtree(drive_models_path)
    shutil.copytree('/content/models', drive_models_path)
    print(f"✓ Models backed up to Google Drive: {drive_models_path}")
    print("\nYour models are now safe even if Colab session ends!")
else:
    print("⚠️ No models directory found")


In [ ]:
## Step 9: Use Your Trained Models

#After training, you can use the detector in your code:
# Example: Protect a database query
def execute_safe_query(query):
    """Execute a query only if it's safe"""
    result = detector.detect(query)
    
    if result['decision'] == 'BLOCK':
        raise ValueError(
            f"SQL injection detected! "
            f"Confidence: {result['confidence']:.2%}, "
            f"Stage: {result['stage']}"
        )
    
    # Safe to execute
    print(f"✅ Query is safe. Executing...")
    # return db.execute(query)  # Uncomment when you have a real DB
    return "Query executed successfully"

# Test it
print("Testing safe query:")
try:
    execute_safe_query("SELECT * FROM users WHERE id = 1")
except ValueError as e:
    print(f"❌ {e}")

print("\nTesting injection query:")
try:
    execute_safe_query("SELECT * FROM users WHERE id = 1 OR 1=1")
except ValueError as e:
    print(f"✅ Blocked: {e}")

## 📊 Summary

✅ **Training Complete!**

Your SQL Injection Detector is now trained and ready to use!

### What You Have:
- ✅ Stage 0: Bloom Filter (whitelist)
- ✅ Stage 1: Linear SVM (keyword detection)
- ✅ Stage 2: DistilBERT (context analysis)

### Models Location:
- `/content/models/` (in Colab)
- `/content/drive/MyDrive/sql_injection_models/` (backed up to Drive)

### Next Steps:
1. Download models from Drive if needed
2. Integrate detector into your application
3. Monitor performance and retrain as needed

### Tips:
- Use more training data (1000+ queries per class) for better accuracy
- Retrain periodically as attack patterns evolve
- Monitor false positives and adjust thresholds in config.yaml

---

**Happy detecting! 🚀**